In [7]:
"""
UNIQUE CONTRIBUTIONS — what makes this project stand out:

1. TWO-LEVEL DETECTION (point-level AND run/job-level)
   → Real ATLAS monitoring flags JOBS, not individual timesteps

2. EARLY DETECTION LATENCY
   → How many seconds until each model catches the anomaly?
   → Critical for grid computing: catch problems early = save CPU-hours

3. ANOMALY FINGERPRINTING
   → Each anomaly type has a unique "signature" in feature space
   → Radar/spider chart per anomaly type

4. RUN-LEVEL CLASSIFICATION with feature aggregation
   → Aggregate per-run stats → classify entire job
   → More realistic than point-level for batch computing

5. FALSE POSITIVE COST ANALYSIS
   → In ATLAS: FP = wasted grid job restart = real cost
   → Show precision vs recall trade-off with cost framing
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              roc_auc_score, roc_curve, confusion_matrix)
from scipy.stats import median_abs_deviation
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 11
PALETTE = ['#1565C0','#E53935','#2E7D32','#F57F17','#6A1B9A','#00838F']

# ─────────────────────────────────────────────────────────
# LOAD & ENGINEER FEATURES
# ─────────────────────────────────────────────────────────
df = pd.read_csv('../data/full_dataset.csv')
df = df.sort_values(['run_id','wtime']).reset_index(drop=True)

base_features = ['pss','vmem','rss','nthreads','wchar','rchar','nprocs']

for feat in ['pss','nthreads','wchar']:
    df[f'{feat}_roll_mean'] = df.groupby('run_id')[feat].transform(
        lambda x: x.rolling(5, min_periods=1).mean())
    df[f'{feat}_roll_std']  = df.groupby('run_id')[feat].transform(
        lambda x: x.rolling(5, min_periods=1).std().fillna(0))
    df[f'{feat}_roc']       = df.groupby('run_id')[feat].transform(
        lambda x: x.diff().fillna(0))

df['mem_ratio']    = df['pss']   / (df['vmem'] + 1e-9)
df['io_intensity'] = (df['wchar'] + df['rchar']) / (df['wtime'] + 1e-9)

eng_features = (base_features +
                [f'{f}_roll_mean' for f in ['pss','nthreads','wchar']] +
                [f'{f}_roll_std'  for f in ['pss','nthreads','wchar']] +
                [f'{f}_roc'       for f in ['pss','nthreads','wchar']] +
                ['mem_ratio','io_intensity'])

df_normal    = df[df['label'] == 0]
y_true       = df['label'].values
contamination = df['label'].mean()

scaler = StandardScaler()
X_train = scaler.fit_transform(df_normal[base_features].fillna(0))
X_all   = scaler.transform(df[base_features].fillna(0))

# Fit models
iso = IsolationForest(n_estimators=200, contamination=contamination, random_state=42)
iso.fit(X_train)
df['iso_score'] = -iso.decision_function(X_all)
df['iso_pred']  = (iso.predict(X_all) == -1).astype(int)

lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination, novelty=True)
lof.fit(X_train)
df['lof_score'] = -lof.decision_function(X_all)
df['lof_pred']  = (lof.predict(X_all) == -1).astype(int)

# Z-Score
z_scores = pd.DataFrame(index=df.index)
for feat in base_features:
    mu, sig = df_normal[feat].mean(), df_normal[feat].std()
    z_scores[feat] = (df[feat] - mu) / (sig + 1e-9)
df['zscore_score'] = z_scores.abs().max(axis=1)
df['zscore_pred']  = (z_scores.abs() > 3.0).any(axis=1).astype(int)

# MAD
mad_scores = pd.DataFrame(index=df.index)
for feat in base_features:
    med = np.median(df_normal[feat])
    mad = median_abs_deviation(df_normal[feat])
    mad_scores[feat] = 0.6745 * (df[feat] - med) / (mad + 1e-9)
df['mad_score'] = mad_scores.abs().max(axis=1)
df['mad_pred']  = (mad_scores.abs() > 3.5).any(axis=1).astype(int)

# Ensemble
preds_all   = ['zscore_pred','mad_pred','iso_pred','lof_pred']
df['ensemble_votes'] = df[preds_all].sum(axis=1)
df['ensemble_pred']  = (df['ensemble_votes'] >= 2).astype(int)
score_norm = df[['zscore_score','mad_score','iso_score','lof_score']].copy()
for c in score_norm.columns:
    mn, mx = score_norm[c].min(), score_norm[c].max()
    score_norm[c] = (score_norm[c] - mn) / (mx - mn + 1e-9)
df['ensemble_score'] = score_norm.mean(axis=1)

models = {
    'Z-Score':         ('zscore_pred', 'zscore_score'),
    'MAD Z-Score':     ('mad_pred',    'mad_score'),
    'Isolation Forest':('iso_pred',    'iso_score'),
    'LOF':             ('lof_pred',    'lof_score'),
    'Ensemble':        ('ensemble_pred','ensemble_score'),
}

print("Models fitted ✓")

# ═══════════════════════════════════════════════════════════
# UNIQUE PLOT 1: ANOMALY FINGERPRINTS (Radar Charts)
# Each anomaly type has a unique multi-dimensional signature
# ═══════════════════════════════════════════════════════════
print("[1] Anomaly Fingerprints radar chart...")

radar_features = ['pss','nthreads','wchar','rchar','nprocs','vmem']
atypes = ['baseline_mem','mem_spike','thread_explosion','io_storm','combined']
atype_labels = ['Baseline','Mem Spike','Thread Expl.','I/O Storm','Combined']
colors_radar  = ['#1565C0','#E53935','#FF9800','#2E7D32','#9C27B0']

# Normalise each feature to 0-1 across all data
scaler_mm = MinMaxScaler()
radar_scaled = pd.DataFrame(
    scaler_mm.fit_transform(df[radar_features]),
    columns=radar_features
)
radar_scaled['anomaly_type'] = df['anomaly_type'].values
radar_means = radar_scaled.groupby('anomaly_type')[radar_features].mean()

N     = len(radar_features)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, axes = plt.subplots(1, 5, figsize=(18, 4),
                         subplot_kw=dict(polar=True))
fig.suptitle('Anomaly Fingerprints — Normalised Feature Signatures',
             fontsize=14, fontweight='bold', y=1.02)

for ax, atype, label, color in zip(axes, atypes, atype_labels, colors_radar):
    if atype not in radar_means.index:
        continue
    vals = radar_means.loc[atype].tolist()
    vals += vals[:1]

    ax.plot(angles, vals, color=color, linewidth=2)
    ax.fill(angles, vals, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_features, fontsize=8)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['','',''], fontsize=6)
    ax.set_ylim(0, 1)
    ax.set_title(label, fontsize=11, fontweight='bold', color=color, pad=12)
    ax.grid(color='grey', alpha=0.3)

plt.tight_layout()
import os

output_dir = os.path.expanduser('~/Atlas_gsoc26/outputs')
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, 'U1_anomaly_fingerprints.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved:", os.path.join(output_dir, 'U1_anomaly_fingerprints.png'))

print("   Saved: U1_anomaly_fingerprints.png")

# ═══════════════════════════════════════════════════════════
# UNIQUE PLOT 2: EARLY DETECTION LATENCY
# How many seconds until each model flags the anomaly?
# Critical insight for real-time ATLAS monitoring
# ═══════════════════════════════════════════════════════════
print("[2] Early detection latency...")

def detection_latency(df, pred_col, atype, max_wtime=None):
    """Return list of wtime values when each run was first flagged."""
    runs = df[df['anomaly_type']==atype]['run_id'].unique()
    latencies = []
    for rid in runs:
        sub = df[df['run_id']==rid].sort_values('wtime')
        if max_wtime is not None:
            sub = sub[sub['wtime'] <= max_wtime]
        flagged = sub[sub[pred_col] == 1]
        if len(flagged):
            latencies.append(flagged.iloc[0]['wtime'])
        else:
            latencies.append(np.nan)   # missed
    return latencies

atypes_detect = ['mem_spike','thread_explosion','io_storm','combined']
model_names   = list(models.keys())

fig, axes = plt.subplots(1, len(atypes_detect), figsize=(16, 5), sharey=False)
fig.suptitle('Early Detection Latency per Model\n(seconds until first flag — lower is better)',
             fontsize=13, fontweight='bold')

for ax, atype in zip(axes, atypes_detect):
    data_for_box = []
    labels_for_box = []
    for mname, (pred_col, _) in models.items():
        lats = [l for l in detection_latency(df, pred_col, atype) if not np.isnan(l)]
        if lats:
            data_for_box.append(lats)
            labels_for_box.append(mname)

    bp = ax.boxplot(data_for_box, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], PALETTE[:len(data_for_box)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_xticklabels(labels_for_box, rotation=30, ha='right', fontsize=8)
    ax.set_title(atype.replace('_',' ').title(), fontweight='bold')
    ax.set_ylabel('Detection time (s)' if ax == axes[0] else '')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
import os

output_dir = os.path.expanduser('~/Atlas_gsoc26/outputs')
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, 'U2_anomaly_fingerprints.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved:", os.path.join(output_dir, 'U2_anomaly_fingerprints.png'))
print("   Saved: U2_detection_latency.png")

# ═══════════════════════════════════════════════════════════
# UNIQUE PLOT 3: RUN-LEVEL vs POINT-LEVEL DETECTION
# Real monitoring flags JOBS, not individual timesteps
# Show that aggregating per-run improves reliability
# ═══════════════════════════════════════════════════════════
print("[3] Run-level vs point-level detection...")

# Aggregate per run: a run is flagged if >threshold% of its points flagged
def run_level_metrics(df, pred_col, score_col, threshold=0.3):
    """Aggregate point predictions to run level."""
    run_stats = df.groupby('run_id').agg(
        true_label  = ('label',      'max'),
        frac_flagged = (pred_col,    'mean'),
        mean_score   = (score_col,   'mean'),
        max_score    = (score_col,   'max'),
    ).reset_index()
    run_stats['run_pred'] = (run_stats['frac_flagged'] >= threshold).astype(int)
    f1  = f1_score(run_stats['true_label'], run_stats['run_pred'], zero_division=0)
    auc = roc_auc_score(run_stats['true_label'], run_stats['mean_score'])
    return f1, auc, run_stats

comparison_rows = []
for mname, (pred_col, score_col) in models.items():
    # Point-level
    pt_f1  = f1_score(y_true, df[pred_col], zero_division=0)
    pt_auc = roc_auc_score(y_true, df[score_col])
    # Run-level
    rl_f1, rl_auc, _ = run_level_metrics(df, pred_col, score_col)
    comparison_rows.append(dict(Model=mname,
                                Point_F1=pt_f1, Run_F1=rl_f1,
                                Point_AUC=pt_auc, Run_AUC=rl_auc))

comp_df = pd.DataFrame(comparison_rows).set_index('Model')
print(comp_df.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Point-Level vs Run/Job-Level Detection\n'
             '(Run-level = realistic for batch computing)',
             fontsize=13, fontweight='bold')

x    = np.arange(len(comp_df))
w    = 0.35
for ax, (pt_col, rl_col, metric) in zip(
        axes,
        [('Point_F1','Run_F1','F1 Score'),
         ('Point_AUC','Run_AUC','ROC-AUC')]):
    bars1 = ax.bar(x - w/2, comp_df[pt_col], w,
                   label='Point-level', color='#90CAF9', alpha=0.85, edgecolor='#1565C0')
    bars2 = ax.bar(x + w/2, comp_df[rl_col], w,
                   label='Run-level',   color='#EF9A9A', alpha=0.85, edgecolor='#C62828')
    ax.set_xticks(x)
    ax.set_xticklabels(comp_df.index, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric}: Point vs Run Level', fontweight='bold')
    ax.set_ylim(0, 1.12)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for b1, b2 in zip(bars1, bars2):
        ax.text(b1.get_x()+b1.get_width()/2, b1.get_height()+0.01,
                f'{b1.get_height():.2f}', ha='center', fontsize=7.5)
        ax.text(b2.get_x()+b2.get_width()/2, b2.get_height()+0.01,
                f'{b2.get_height():.2f}', ha='center', fontsize=7.5)

plt.tight_layout()
import os

output_dir = os.path.expanduser('~/Atlas_gsoc26/outputs')
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, 'U3_anomaly_fingerprints.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved:", os.path.join(output_dir, 'U3_anomaly_fingerprints.png'))
plt.close()
print("   Saved: U3_run_vs_point_level.png")

# ═══════════════════════════════════════════════════════════
# UNIQUE PLOT 4: PRECISION-RECALL TRADE-OFF WITH COST FRAMING
# In ATLAS: FP = wasted grid restart, FN = undetected bad job
# Show the real operational cost of each model
# ═══════════════════════════════════════════════════════════
print("[4] Precision-Recall with ATLAS cost framing...")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Operational Cost Analysis for ATLAS Grid Monitoring',
             fontsize=13, fontweight='bold')

# Left: PR curve per model
ax = axes[0]
from sklearn.metrics import precision_recall_curve, average_precision_score
for (mname, (_, score_col)), color in zip(models.items(), PALETTE):
    scores = df[score_col].values
    prec, rec, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)
    ax.plot(rec, prec, color=color, linewidth=2, label=f'{mname} (AP={ap:.2f})')

ax.set_xlabel('Recall (FN Rate ↓ = fewer missed anomalies)')
ax.set_ylabel('Precision (FP Rate ↓ = fewer false alarms)')
ax.set_title('Precision-Recall Curves', fontweight='bold')
ax.legend(fontsize=8, loc='lower left')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)

# Right: Cost matrix heatmap
# Assume: FP costs 1 CPU-hour (wasted restart), FN costs 10 CPU-hours (undetected failure)
ax2 = axes[1]
FP_COST = 1    # CPU-hours wasted per false alarm
FN_COST = 10   # CPU-hours lost per missed anomaly

cost_rows = []
for mname, (pred_col, _) in models.items():
    preds = df[pred_col].values
    cm    = confusion_matrix(y_true, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (0,0,0,0)
    total_cost = fp * FP_COST + fn * FN_COST
    cost_rows.append(dict(Model=mname, TN=tn, FP=fp, FN=fn, TP=tp,
                          FP_cost=fp*FP_COST, FN_cost=fn*FN_COST,
                          Total_cost=total_cost))

cost_df = pd.DataFrame(cost_rows).set_index('Model')

bars = ax2.barh(cost_df.index,
                cost_df['Total_cost'],
                color=[PALETTE[i] for i in range(len(cost_df))],
                alpha=0.85)
ax2.barh(cost_df.index, cost_df['FP_cost'],
         color='#90CAF9', alpha=0.6, label=f'FP cost ({FP_COST} CPU-hr/FP)')
ax2.barh(cost_df.index, cost_df['FN_cost'],
         left=cost_df['FP_cost'],
         color='#EF9A9A', alpha=0.8, label=f'FN cost ({FN_COST} CPU-hr/FN)')

ax2.set_xlabel('Estimated CPU-hours lost')
ax2.set_title(f'Operational Cost\n(FP={FP_COST} hr, FN={FN_COST} hr)',
              fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='x', alpha=0.3)
for bar, row in zip(ax2.patches[:len(cost_df)], cost_df.itertuples()):
    ax2.text(cost_df['Total_cost'].max() * 0.02,
             bar.get_y() + bar.get_height()/2,
             f'{row.Total_cost:.0f} hr',
             va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
import os

output_dir = os.path.expanduser('~/Atlas_gsoc26/outputs')
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, 'U4_anomaly_fingerprints.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved:", os.path.join(output_dir, 'U4_anomaly_fingerprints.png'))
plt.close()
print("   Saved: U4_cost_analysis.png")

# ═══════════════════════════════════════════════════════════
# UNIQUE PLOT 5: FEATURE CORRELATION NETWORK
# Show which features move together — motivates feature selection
# ═══════════════════════════════════════════════════════════
print("[5] Feature correlation heatmap (normal vs anomalous)...")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Feature Correlations: Normal vs Anomalous Runs\n'
             '(structural differences reveal anomaly signatures)',
             fontsize=13, fontweight='bold')

plot_features = ['pss','rss','vmem','nthreads','wchar','rchar','nprocs']
df_anom = df[df['label'] == 1]

for ax, subset, title in [
    (axes[0], df_normal, 'Normal Runs'),
    (axes[1], df_anom,   'Anomalous Runs')
]:
    corr = subset[plot_features].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=ax, annot=True, fmt='.2f',
                cmap='RdBu_r', vmin=-1, vmax=1,
                mask=mask, square=True, linewidths=0.5,
                cbar_kws={'shrink': 0.8})
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
import os

output_dir = os.path.expanduser('~/Atlas_gsoc26/outputs')
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, 'U5_anomaly_fingerprints.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved:", os.path.join(output_dir, 'U5_anomaly_fingerprints.png'))
print("   Saved: U5_correlation_shift.png")

# ═══════════════════════════════════════════════════════════
# UNIQUE PLOT 6: MONITORING DASHBOARD SIMULATION
# What a real ATLAS operator would see in production
# ═══════════════════════════════════════════════════════════
print("[6] Monitoring dashboard simulation...")

# Pick one run of each type to simulate live monitoring
run_examples = {}
for atype in ['baseline_mem','mem_spike','thread_explosion','io_storm']:
    run_examples[atype] = df[df['anomaly_type']==atype]['run_id'].iloc[0]

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Simulated ATLAS Job Monitoring Dashboard\n'
             '(Ensemble anomaly score in real-time)',
             fontsize=14, fontweight='bold')

plot_configs = [
    ('baseline_mem',    '#1565C0', 'Normal Job'),
    ('mem_spike',       '#E53935', 'Memory Spike'),
    ('thread_explosion','#FF9800', 'Thread Explosion'),
    ('io_storm',        '#2E7D32', 'I/O Storm'),
]

for idx, (atype, color, title) in enumerate(plot_configs):
    rid  = run_examples[atype]
    sub  = df[df['run_id']==rid].sort_values('wtime')

    ax1 = fig.add_subplot(4, 3, idx*3 + 1)
    ax2 = fig.add_subplot(4, 3, idx*3 + 2)
    ax3 = fig.add_subplot(4, 3, idx*3 + 3)

    # PSS memory
    ax1.fill_between(sub['wtime'], sub['pss'], alpha=0.4, color=color)
    ax1.plot(sub['wtime'], sub['pss'], color=color, linewidth=1.5)
    ax1.axhline(df_normal['pss'].mean() + 3*df_normal['pss'].std(),
                color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax1.set_ylabel('PSS (MB)')
    ax1.set_title(f'{title}', fontweight='bold', color=color)
    ax1.grid(alpha=0.3)

    # Thread count
    ax2.step(sub['wtime'], sub['nthreads'], color=color, linewidth=1.5, where='post')
    ax2.axhline(df_normal['nthreads'].mean() + 3*df_normal['nthreads'].std(),
                color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax2.set_ylabel('Threads')
    ax2.grid(alpha=0.3)

    # Ensemble score — the key output
    ax3.fill_between(sub['wtime'], sub['ensemble_score'],
                     alpha=0.35, color=color)
    ax3.plot(sub['wtime'], sub['ensemble_score'], color=color, linewidth=2)
    ax3.axhline(0.5, color='red', linestyle='--', linewidth=1.5,
                label='Alert threshold')
    # Shade alert zone
    ax3.axhspan(0.5, 1.0, alpha=0.08, color='red')
    ax3.set_ylim(0, 1)
    ax3.set_ylabel('Anomaly Score')
    ax3.set_xlabel('Wall time (s)')
    ax3.legend(fontsize=7, loc='upper left')
    ax3.grid(alpha=0.3)

    # Add column headers on first row
    if idx == 0:
        ax1.set_title('Memory (PSS)', fontsize=10, color='grey')
        ax2.set_title('Thread Count', fontsize=10, color='grey')
        ax3.set_title('Ensemble Anomaly Score ◄ KEY OUTPUT',
                      fontsize=10, color='grey')

plt.tight_layout()
import os

output_dir = os.path.expanduser('~/Atlas_gsoc26/outputs')
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, 'U6_anomaly_fingerprints.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved:", os.path.join(output_dir, 'U6_anomaly_fingerprints.png'))
print("   Saved: U6_monitoring_dashboard.png")

print("\n✓ All 6 unique plots generated!")
print("\n=== SUMMARY OF UNIQUE CONTRIBUTIONS ===")
print("U1  Anomaly Fingerprints    - radar charts show each type's signature")
print("U2  Detection Latency       - seconds until each model fires")
print("U3  Run-level Detection     - job-level (realistic) vs point-level")
print("U4  Cost Analysis           - FP/FN in CPU-hours (ATLAS-specific)")
print("U5  Correlation Shift       - how feature relationships change")
print("U6  Monitoring Dashboard    - what an operator would actually see")

Models fitted ✓
[1] Anomaly Fingerprints radar chart...
Saved: /home/chaim/Atlas_gsoc26/outputs/U1_anomaly_fingerprints.png
   Saved: U1_anomaly_fingerprints.png
[2] Early detection latency...
Saved: /home/chaim/Atlas_gsoc26/outputs/U2_anomaly_fingerprints.png
   Saved: U2_detection_latency.png
[3] Run-level vs point-level detection...
                  Point_F1  Run_F1  Point_AUC  Run_AUC
Model                                                 
Z-Score              0.940   0.821      1.000    1.000
MAD Z-Score          0.709   0.719      0.939    0.715
Isolation Forest     0.765   0.748      0.920    0.830
LOF                  0.924   0.807      0.971    0.811
Ensemble             0.791   0.760      0.966    0.911
Saved: /home/chaim/Atlas_gsoc26/outputs/U3_anomaly_fingerprints.png
   Saved: U3_run_vs_point_level.png
[4] Precision-Recall with ATLAS cost framing...
Saved: /home/chaim/Atlas_gsoc26/outputs/U4_anomaly_fingerprints.png
   Saved: U4_cost_analysis.png
[5] Feature correlation he